# 04. Inference: real-time endpoint with Data Capture

1. **Endpoint (BP-701):** deploy the **approved** model package to an `ml.m5.large` endpoint, with Data Capture writing every request and response to S3.
2. **Encode records (BP-702):** raw customer records -> CSV rows, using the saved `feature_schema.json` (the same fitted features as training).
3. **Invoke (BP-703):** score 200 "new" customers, the newest rows of the test period.
4. **Captured data (BP-704):** confirm the capture files in `inference-capture/`.

**Cost:** an endpoint is billed every hour it runs, so delete it with the clean-up section of notebook 05 once you have your screenshots.

In [ ]:
# Setup: repository root, config, lab connection and the state saved by earlier notebooks.
from pathlib import Path

import pandas as pd
import sagemaker

from bankpulse_core.aws import lab_boto_session, read_state, s3_uri, update_state
from bankpulse_core.config import load_config

ROOT = Path.cwd() if (Path.cwd() / 'config').is_dir() else Path.cwd().parent
config = load_config(ROOT / 'config' / 'data_prep.yaml')
STATE = Path(config['output']['processed_dir']) / 'sagemaker_state.json'
state = read_state(STATE)

boto_session = lab_boto_session()                       # bankpulse-lab profile; no keys in code
sm_session = sagemaker.Session(boto_session=boto_session)
sm_client = boto_session.client('sagemaker')
bucket, prefix, role = state['bucket'], state['prefix'], state['role']

import re
import time

from bankpulse_core.io import list_objects, read_bytes, read_csv, read_json
from bankpulse_core.pipeline import encode_records, to_csv_payload

assert state.get('approval_status') == 'Approved', 'Run notebook 03 first: only an approved model is deployed'
print('model package:', state['model_package_arn'].split('/', 1)[1])

## BP-701. Endpoint with Data Capture

Deploying creates three linked SageMaker resources:
- **Model:** the container plus artifact, taken from the approved model package.
- **Endpoint configuration:** the instance type and count, plus the **Data Capture** settings (100% of requests and responses, written to S3).
- **Endpoint:** the HTTPS service that runs the configuration. It takes about 6-8 minutes to reach **InService**.

**Screenshots:** the endpoint **InService**, and the endpoint configuration showing Data capture enabled and its S3 location.

In [ ]:
from sagemaker import ModelPackage
from sagemaker.model_monitor import DataCaptureConfig

ENDPOINT = f'bankpulse-term-deposit-{time.strftime("%Y%m%d-%H%M%S")}'
capture_uri = s3_uri(bucket, prefix, 'inference-capture')

# Step 1: the model to deploy is the approved registry version, not a raw artifact.
model = ModelPackage(role=role, model_package_arn=state['model_package_arn'], sagemaker_session=sm_session)

# Step 2: capture every request and response (100%) into the data lake.
capture = DataCaptureConfig(enable_capture=True, sampling_percentage=100, destination_s3_uri=capture_uri,
                            capture_options=['REQUEST', 'RESPONSE'], csv_content_types=['text/csv'])

# Step 3: create the model, endpoint config and endpoint, and wait until it is InService.
model.deploy(initial_instance_count=1, instance_type='ml.m5.large', endpoint_name=ENDPOINT,
             data_capture_config=capture, wait=True)
state = update_state(STATE, endpoint=ENDPOINT, capture_uri=capture_uri)

endpoint = sm_client.describe_endpoint(EndpointName=ENDPOINT)
display(pd.Series({'endpoint': ENDPOINT, 'status': endpoint['EndpointStatus'],
                   'endpoint_config': endpoint['EndpointConfigName'],
                   'data_capture': endpoint['DataCaptureConfig']['EnableCapture'],
                   'capture_destination': endpoint['DataCaptureConfig']['DestinationS3Uri'].split(f'{bucket}/', 1)[1]
                   }).to_frame('endpoint'))

## BP-702. Encode raw records with `feature_schema.json`

The endpoint expects the **same 73 numbers, in the same order** as the training files. `encode_records` rebuilds each feature from the schema with its saved training state (caps, category lists), so a live customer is encoded exactly like a training row. A test checks it is byte-identical to the batch files.

The "new" customers are the **newest 200 rows** of the dataset, with the label and `duration` removed, as an application would send them.

In [ ]:
schema = read_json(state['s3']['feature_schema'])
raw = read_csv(config['data']['path'], sep=config['data']['separator'])
truth = raw['y'].iloc[-200:].eq('yes').astype(int).to_numpy()             # kept aside, never sent
new_customers = raw.iloc[-200:].drop(columns=['y', 'duration'])

rows = encode_records(new_customers.to_dict('records'), schema)
print(rows.shape[0], 'customers ->', rows.shape[1], 'model inputs each')
print('first CSV row sent:', to_csv_payload(rows.iloc[:1]).strip()[:120], '...')

## BP-703. Invoke the endpoint

`InvokeEndpoint` sends a `text/csv` body (one customer per line) and gets back one probability per line. The customers are sent in batches of 20, so Data Capture records 10 separate requests.

**Screenshot:** the table of scored customers below.

In [ ]:
runtime = boto_session.client('sagemaker-runtime')
threshold = float(state['threshold'])

scores = []
for start in range(0, len(rows), 20):
    response = runtime.invoke_endpoint(EndpointName=ENDPOINT, ContentType='text/csv', Accept='text/csv',
                                       Body=to_csv_payload(rows.iloc[start:start + 20]))
    body = response['Body'].read().decode('utf-8')
    scores += [float(value) for value in re.split(r'[,\n]', body) if value.strip()]

scored = new_customers[['age', 'job', 'contact', 'month', 'campaign', 'poutcome']].assign(
    score=[round(s, 4) for s in scores], call_first=[s >= threshold for s in scores], subscribed=truth)
display(scored.sort_values('score', ascending=False).head(10))
print(f'{len(scores)} customers scored; {int(scored.call_first.sum())} above the threshold {threshold:.3f}')

In [ ]:
# The live endpoint must agree with Batch Transform for the same customers (same model, same encoding).
from bankpulse_core.evaluation import parse_batch_output

batch_scores = parse_batch_output(state['batch_predictions']['test'])[-200:]
print('largest difference endpoint vs batch:', float(abs(pd.Series(scores) - batch_scores).max()))

## BP-704. Confirm the captured data

Data Capture buffers requests and writes them to S3 as JSON Lines, usually within a few minutes, under:
`inference-capture/<endpoint>/<variant>/<yyyy>/<mm>/<dd>/<hh>/`

**Screenshots:** the `.jsonl` files in S3, and one capture record opened.

In [ ]:
# Wait up to 10 minutes for the first capture files to arrive.
prefix_uri = s3_uri(bucket, prefix, 'inference-capture', ENDPOINT) + '/'
for _ in range(40):
    capture_files = [uri for uri in list_objects(prefix_uri) if uri.endswith('.jsonl')]
    if capture_files:
        break
    time.sleep(15)
assert capture_files, 'no capture files yet: re-run this cell in a few minutes'
state = update_state(STATE, capture_files=capture_files)
display(pd.DataFrame({'capture file': [uri.split(f'{bucket}/', 1)[1] for uri in capture_files]}))

In [ ]:
# One captured request: the CSV rows sent, and the probabilities returned.
import json

record = json.loads(read_bytes(capture_files[0]).decode('utf-8').splitlines()[0])
data = record['captureData']
print('event time :', record['eventMetadata']['inferenceTime'])
print('input      :', data['endpointInput']['data'].splitlines()[0][:100], '...')
print('output     :', data['endpointOutput']['data'].replace('\n', ' ')[:100], '...')